# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 04 — Linear SVM

---

### Purpose
Train and evaluate a Linear SVM classifier (LinearSVC + CalibratedClassifierCV)
on all four feature sets. Linear SVMs are historically one of the strongest
text classification baselines.

### Objectives
1. Load all feature matrices
2. Train Linear SVM on each feature set
3. Evaluate with full metrics suite
4. Generate confusion matrices, classification reports, ROC curves
5. Inspect decision function and support vector structure
6. Save trained models
7. Show prediction examples

### Notebook Outline
1. Imports
2. Configuration
3. Load TF-IDF Features
4. Load Character N-Gram Features
5. Load Stylometric Features
6. Load Embedding Features
7. Training
8. Evaluation
9. Confusion Matrix
10. Classification Report
11. ROC Curves
12. Decision Function Inspection
13. Model Saving
14. Prediction Examples
15. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import plotly.express as px

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.classifiers import LinearSVMClassifier
from src.feature_engineering.utils import load_feature_matrix, setup_logger
from src.evaluation.evaluator import ModelEvaluator
from src.visualization.plots import (
    plot_confusion_matrix, plot_roc_curves, plot_feature_importance
)
from src.utils.helpers import (
    set_global_seed, train_test_val_split, load_yaml, make_output_dirs,
    display_metrics_table, print_section_header,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED  = cfg['random_seed']
TEST_SIZE    = cfg['evaluation']['test_size']
VAL_SIZE     = cfg['evaluation']['val_size']
SVM_CFG      = cfg['linear_svm']
FEAT_CFG     = cfg['features']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

print(f'SVM Config : {SVM_CFG}')

---

## 3. Load TF-IDF Features

In [ ]:
X_tfidf, y_tfidf = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['tfidf']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)
classes = np.load(
    str(PROJECT_ROOT / 'data' / 'features' / 'tfidf' / 'classes_tfidf_fingerprint.npy'),
    allow_pickle=True,
)
X_tr_tf, X_val_tf, X_te_tf, y_tr_tf, y_val_tf, y_te_tf = train_test_val_split(
    X_tfidf, y_tfidf, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'TF-IDF → Train {X_tr_tf.shape[0]}  Test {X_te_tf.shape[0]}')

---

## 4. Load Character N-Gram Features

In [ ]:
X_char, y_char = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['char']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)
X_tr_ch, X_val_ch, X_te_ch, y_tr_ch, y_val_ch, y_te_ch = train_test_val_split(
    X_char, y_char, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Char N-Gram → Train {X_tr_ch.shape[0]}  Test {X_te_ch.shape[0]}')

---

## 5. Load Stylometric Features

In [ ]:
X_style, y_style = load_feature_matrix(
    PROJECT_ROOT / FEAT_CFG['style']['fingerprint'],
    PROJECT_ROOT / FEAT_CFG['labels']['fingerprint'],
)
if sp.issparse(X_style):
    X_style = X_style.toarray()

X_tr_st, X_val_st, X_te_st, y_tr_st, y_val_st, y_te_st = train_test_val_split(
    X_style, y_style, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Stylometric → Train {X_tr_st.shape[0]}  Test {X_te_st.shape[0]}')

---

## 6. Load Embedding Features

In [ ]:
EMB_DIR  = PROJECT_ROOT / 'data' / 'features' / 'embedding'
X_emb    = np.load(str(EMB_DIR / 'emb_fingerprint.npz'))['embeddings']
y_emb    = np.load(str(EMB_DIR / 'labels_emb_fingerprint.npy'))

X_tr_em, X_val_em, X_te_em, y_tr_em, y_val_em, y_te_em = train_test_val_split(
    X_emb, y_emb, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_SEED
)
print(f'Embeddings → Train {X_tr_em.shape[0]}  Test {X_te_em.shape[0]}')

---

## 7. Training

In [ ]:
def run_linear_svm(X_train, X_test, y_train, y_test, feature_set: str):
    """Train a Linear SVM classifier and return (model, evaluator)."""
    model = LinearSVMClassifier(cfg=SVM_CFG)
    model.fit(X_train, y_train, feature_set=feature_set)

    evaluator = ModelEvaluator(
        model_name='linear_svm',
        feature_set=feature_set,
        classes=classes,
    )
    evaluator.evaluate(
        estimator=model.model,
        X_test=X_test,
        y_test=y_test,
        train_time=model.train_time_,
    )
    return model, evaluator

In [ ]:
print_section_header('Training Linear SVM — TF-IDF Features')
svm_tfidf, eval_svm_tfidf = run_linear_svm(X_tr_tf, X_te_tf, y_tr_tf, y_te_tf, 'tfidf')

print_section_header('Training Linear SVM — Char N-Gram Features')
svm_char, eval_svm_char = run_linear_svm(X_tr_ch, X_te_ch, y_tr_ch, y_te_ch, 'char')

print_section_header('Training Linear SVM — Stylometric Features')
svm_style, eval_svm_style = run_linear_svm(X_tr_st, X_te_st, y_tr_st, y_te_st, 'style')

print_section_header('Training Linear SVM — Embedding Features')
svm_emb, eval_svm_emb = run_linear_svm(X_tr_em, X_te_em, y_tr_em, y_te_em, 'embedding')

print('\n✅ All Linear SVM variants trained.')

---

## 8. Evaluation

In [ ]:
results_svm = pd.DataFrame([
    eval_svm_tfidf.to_series(),
    eval_svm_char.to_series(),
    eval_svm_style.to_series(),
    eval_svm_emb.to_series(),
])

display_cols = [
    'model_name', 'feature_set', 'accuracy',
    'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted',
    'roc_auc_macro', 'train_time_s', 'pred_time_s',
]
results_svm[display_cols].style.highlight_max(
    subset=['accuracy', 'f1_macro'],
    color='lightgreen'
).format(precision=4)

---

## 9. Confusion Matrix

In [ ]:
for evaluator, label in [
    (eval_svm_tfidf, 'TF-IDF'),
    (eval_svm_char,  'Char N-Gram'),
    (eval_svm_style, 'Stylometric'),
    (eval_svm_emb,   'Embedding'),
]:
    cm = np.array(evaluator.results_['confusion_matrix'])
    out_path = DIR_FIGURES / f'svm_cm_{evaluator.feature_set}.png'
    plot_confusion_matrix(
        cm=cm, class_names=list(classes),
        title=f'Linear SVM — {label} — Confusion Matrix',
        out_path=out_path, normalize=True,
    )
    print(f'✅ {out_path.name}')

---

## 10. Classification Report

In [ ]:
best_svm_eval = max(
    [eval_svm_tfidf, eval_svm_char, eval_svm_style, eval_svm_emb],
    key=lambda e: e.results_['f1_macro'],
)
print(f'Best SVM feature set: {best_svm_eval.feature_set}')
print(best_svm_eval.results_['classification_report'])

best_svm_eval.save_classification_report(
    DIR_OUTPUTS / f'svm_{best_svm_eval.feature_set}_classification_report.txt'
)

---

## 11. ROC Curves

In [ ]:
if best_svm_eval.results_.get('y_proba') is not None:
    roc_path = DIR_FIGURES / f'svm_roc_{best_svm_eval.feature_set}.png'
    plot_roc_curves(
        y_test=best_svm_eval.results_['y_test'],
        y_proba=best_svm_eval.results_['y_proba'],
        class_names=list(classes),
        title=f'Linear SVM — {best_svm_eval.feature_set} — ROC Curves',
        out_path=roc_path,
    )
    print(f'✅ ROC curves saved: {roc_path.name}')

---

## 12. Decision Function Inspection

In [ ]:
# ── Inspect LinearSVC weight vectors ──────────────────────────────────────────
# The underlying LinearSVC weights can be accessed via .base_estimator
# (since we wrapped in CalibratedClassifierCV)

base_svm = svm_tfidf.model.base_estimator   # LinearSVC
if hasattr(base_svm, 'coef_'):
    coef_matrix = base_svm.coef_
    mean_abs_coef = np.abs(coef_matrix).mean(axis=0)

    from src.feature_engineering.tfidf_extractor import TFIDFExtractor
    tfidf_extractor = TFIDFExtractor.load(PROJECT_ROOT / 'data' / 'features' / 'tfidf')
    word_feature_names = list(tfidf_extractor.get_feature_names('word'))

    coef_path = DIR_FIGURES / 'svm_tfidf_top_features.png'
    plot_feature_importance(
        importances=mean_abs_coef,
        feature_names=word_feature_names,
        title='Linear SVM — Top TF-IDF Features (Mean |Weight|)',
        out_path=coef_path,
        top_n=30,
    )
    print(f'✅ Feature weight chart saved: {coef_path.name}')
else:
    print('Weight inspection not available for CalibratedClassifierCV.')

---

## 13. Model Saving

In [ ]:
for model, suffix in [
    (svm_tfidf, 'tfidf'),
    (svm_char,  'char'),
    (svm_style, 'style'),
    (svm_emb,   'embedding'),
]:
    saved_path = model.save(DIR_MODELS / 'linear_svm', suffix=suffix)
    print(f'✅ Saved: {saved_path.name}')

---

## 14. Prediction Examples

In [ ]:
best_svm = {  # map feature_set → (model, X_test, y_test)
    'tfidf':     (svm_tfidf, X_te_tf, y_te_tf),
    'char':      (svm_char,  X_te_ch, y_te_ch),
    'style':     (svm_style, X_te_st, y_te_st),
    'embedding': (svm_emb,   X_te_em, y_te_em),
}[best_svm_eval.feature_set]

model, X_te, y_te = best_svm
y_pred_sample  = model.predict(X_te[:10])
y_proba_sample = model.predict_proba(X_te[:10])

pd.DataFrame({
    'True Label':      [classes[i] for i in y_te[:10]],
    'Predicted Label': [classes[i] for i in y_pred_sample],
    'Correct':         y_te[:10] == y_pred_sample,
    'Max Confidence':  np.max(y_proba_sample, axis=1).round(4) if y_proba_sample is not None else ['N/A'] * 10,
})

---

## 15. Notebook Summary

### Linear SVM Results Summary

| Feature Set | Macro F1 | Weighted F1 | ROC-AUC |
|---|---|---|---|
| TF-IDF | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Char N-Gram | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Stylometric | *(run to populate)* | *(run to populate)* | *(run to populate)* |
| Embeddings | *(run to populate)* | *(run to populate)* | *(run to populate)* |

### Key Observations (to be completed after execution)
- Best feature set for Linear SVM: **TBD**
- Comparison to Logistic Regression: **TBD**

→ **Notebook 05**: Random Forest

---
*Fingerprint Project — Linear SVM — Complete*